# Adversarially Regularized Variational Graph Autoencoder (ARGVA)

Node Clustering on Cora (Planetoid): Graph representation learning and clustering via adversarial variational autoencoding. This notebook implements the approach with `ARGVA` inside a `Encoder` model, trained with the Adam optimizer, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `ARGVA` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

import os
import os.path as osp

# Switch to 'torch', 'tensorflow', or 'jax'
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import matplotlib.pyplot as plt
import numpy as np
import keras
from keras import layers, ops
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics.cluster import (
    completeness_score,
    homogeneity_score,
    v_measure_score,
)

# K3-Node multi-backend imports
import k3_node.transforms as T
from k3_node.datasets import Planetoid
from k3_node.layers import GCNConv
from k3_node.models import ARGVA
from k3_node.models.utils import negative_sampling

# 1. Dataset & Transforms (framework-agnostic, no ToDevice needed)
path = osp.join(".", "data", "Planetoid")
dataset = Planetoid(path, name="Cora")

transform = T.RandomLinkSplit(
    num_val=0.05,
    num_test=0.1,
    is_undirected=True,
    split_labels=True,
    add_negative_train_samples=False,
)
train_data, val_data, test_data = transform(dataset[0])


# 2. Pure Keras 3 Multi-Backend Model Definitions
class Encoder(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv_mu = GCNConv(hidden_channels, out_channels)
        self.conv_logstd = GCNConv(hidden_channels, out_channels)

    def call(self, x, edge_index):
        x = ops.relu(self.conv1(x, edge_index))
        return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)


class Discriminator(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.lin1 = layers.Dense(hidden_channels)
        self.lin2 = layers.Dense(hidden_channels)
        self.lin3 = layers.Dense(out_channels)

    def call(self, x):
        x = ops.relu(self.lin1(x))
        x = ops.relu(self.lin2(x))
        return self.lin3(x)


# 3. Model Initialization
encoder = Encoder(train_data.num_features, hidden_channels=32, out_channels=32)
discriminator = Discriminator(in_channels=32, hidden_channels=64, out_channels=32)
model = ARGVA(encoder, discriminator)

# Eager forward pass to build weights with backend tensors
_ = encoder(train_data.x, train_data.edge_index)
_ = discriminator(keras.random.normal((1, 32)))

# 4. Standard Keras 3 Optimizers
encoder_optimizer = keras.optimizers.Adam(learning_rate=0.005)
discriminator_optimizer = keras.optimizers.Adam(learning_rate=0.001)

backend = keras.config.backend()

# Setup optimizer variables for functional backends (JAX)
if backend == "jax":
    import jax
    import jax.numpy as jnp

    encoder_optimizer.build(encoder.trainable_variables)
    discriminator_optimizer.build(discriminator.trainable_variables)
    enc_opt_vars = [v.value for v in encoder_optimizer.variables]
    disc_opt_vars = [v.value for v in discriminator_optimizer.variables]
    enc_non_trainable = [v.value for v in encoder.non_trainable_variables]
    disc_non_trainable = [v.value for v in discriminator.non_trainable_variables]
    EPS = 1e-15
    MAX_LOGSTD = 10


# 5. Multi-Backend Training Functions (torch, tensorflow, jax)
def train():
    global enc_opt_vars, disc_opt_vars
    if hasattr(model, "train"):
        model.train()

    pos_edge_index = getattr(
        train_data,
        "pos_edge_label_index",
        getattr(train_data, "edge_label_index", train_data.edge_index),
    )

    # ------------------- PyTorch Backend -------------------
    if backend == "torch":
        z = model.encode(train_data.x, train_data.edge_index)

        for _ in range(5):
            d_loss = model.discriminator_loss(z)
            d_loss.backward()
            d_grads = [v.value.grad for v in discriminator.trainable_variables if v.value.grad is not None]
            discriminator_optimizer.apply_gradients(zip(d_grads, discriminator.trainable_variables))
            for v in discriminator.trainable_variables:
                if v.value.grad is not None:
                    v.value.grad.zero_()

        loss = (
            model.recon_loss(z, pos_edge_index)
            + model.reg_loss(z)
            + (1 / train_data.num_nodes) * model.kl_loss()
        )
        loss.backward()
        enc_grads = [v.value.grad for v in encoder.trainable_variables if v.value.grad is not None]
        encoder_optimizer.apply_gradients(zip(enc_grads, encoder.trainable_variables))
        for v in encoder.trainable_variables:
            if v.value.grad is not None:
                v.value.grad.zero_()
        return float(ops.convert_to_numpy(loss))

    # ------------------- TensorFlow Backend -------------------
    elif backend == "tensorflow":
        import tensorflow as tf

        z = model.encode(train_data.x, train_data.edge_index)
        for _ in range(5):
            with tf.GradientTape() as tape:
                d_loss = model.discriminator_loss(z)
            d_grads = tape.gradient(d_loss, discriminator.trainable_variables)
            discriminator_optimizer.apply_gradients(zip(d_grads, discriminator.trainable_variables))

        with tf.GradientTape() as tape:
            z_enc = model.encode(train_data.x, train_data.edge_index)
            loss = (
                model.recon_loss(z_enc, pos_edge_index)
                + model.reg_loss(z_enc)
                + (1 / train_data.num_nodes) * model.kl_loss()
            )
        enc_grads = tape.gradient(loss, encoder.trainable_variables)
        encoder_optimizer.apply_gradients(zip(enc_grads, encoder.trainable_variables))
        return float(ops.convert_to_numpy(loss))

    # ------------------- JAX Backend -------------------
    elif backend == "jax":
        disc_trainable = [v.value for v in discriminator.trainable_variables]
        enc_trainable = [v.value for v in encoder.trainable_variables]

        def jax_enc_loss_fn(e_vars, d_vars):
            (mu, logstd), _ = encoder.stateless_call(e_vars, enc_non_trainable, train_data.x, train_data.edge_index)
            logstd = jnp.minimum(logstd, MAX_LOGSTD)
            eps = jax.random.normal(jax.random.PRNGKey(0), logstd.shape)
            z_latent = mu + eps * jnp.exp(logstd)

            row, col = pos_edge_index[0], pos_edge_index[1]
            pos_val = jnp.sum(z_latent[row] * z_latent[col], axis=1)
            recon = -jnp.mean(jnp.log(jax.nn.sigmoid(pos_val) + EPS))

            d_fake, _ = discriminator.stateless_call(d_vars, disc_non_trainable, z_latent)
            reg = -jnp.mean(jnp.log(jax.nn.sigmoid(d_fake) + EPS))

            kl = -0.5 * jnp.mean(jnp.sum(1 + 2 * logstd - jnp.square(mu) - jnp.square(jnp.exp(logstd)), axis=1))
            return recon + reg + (1.0 / train_data.num_nodes) * kl, z_latent

        def jax_disc_loss_fn(d_vars, z_in):
            real_noise = jax.random.normal(jax.random.PRNGKey(42), z_in.shape)
            real_out, _ = discriminator.stateless_call(d_vars, disc_non_trainable, real_noise)
            fake_out, _ = discriminator.stateless_call(d_vars, disc_non_trainable, jax.lax.stop_gradient(z_in))
            r_loss = -jnp.mean(jnp.log(jax.nn.sigmoid(real_out) + EPS))
            f_loss = -jnp.mean(jnp.log(1.0 - jax.nn.sigmoid(fake_out) + EPS))
            return r_loss + f_loss

        (loss_val, z_curr), enc_grads = jax.value_and_grad(jax_enc_loss_fn, has_aux=True)(
            enc_trainable, disc_trainable
        )

        for _ in range(5):
            _, disc_grads = jax.value_and_grad(jax_disc_loss_fn)(disc_trainable, z_curr)
            disc_trainable, disc_opt_vars = discriminator_optimizer.stateless_apply(
                disc_opt_vars, disc_grads, disc_trainable
            )

        enc_trainable, enc_opt_vars = encoder_optimizer.stateless_apply(
            enc_opt_vars, enc_grads, enc_trainable
        )

        for v, val in zip(discriminator.trainable_variables, disc_trainable):
            v.assign(val)
        for v, val in zip(encoder.trainable_variables, enc_trainable):
            v.assign(val)

        return float(loss_val)


# 6. Evaluation & Clustering
def test(data):
    if hasattr(model, "eval"):
        model.eval()

    z = model.encode(data.x, data.edge_index)
    kmeans_input = ops.convert_to_numpy(z)
    kmeans = KMeans(n_clusters=7, random_state=0, n_init="auto").fit(kmeans_input)
    pred = kmeans.predict(kmeans_input)

    labels = ops.convert_to_numpy(data.y)
    completeness = completeness_score(labels, pred)
    hm = homogeneity_score(labels, pred)
    nmi = v_measure_score(labels, pred)

    pos_edge = getattr(
        data,
        "pos_edge_label_index",
        getattr(data, "edge_label_index", data.edge_index),
    )
    neg_edge = getattr(data, "neg_edge_label_index", None)
    if neg_edge is None:
        neg_edge = negative_sampling(
            data.edge_index,
            num_nodes=data.num_nodes,
            num_neg_samples=int(ops.shape(pos_edge)[1]),
        )

    auc, ap = model.test(z, pos_edge, neg_edge)
    return auc, ap, completeness, hm, nmi


# 7. Training Loop
for epoch in range(1, 151):
    loss = train()
    auc, ap, completeness, hm, nmi = test(test_data)
    print(
        f"Epoch: {epoch:03d}, Loss: {loss:.3f}, AUC: {auc:.3f}, "
        f"AP: {ap:.3f}, Completeness: {completeness:.3f}, "
        f"Homogeneity: {hm:.3f}, NMI: {nmi:.3f}"
    )


# 8. Visualization (Safe Class Resolution)
def plot_points(data, colors):
    if hasattr(model, "eval"):
        model.eval()
    z = model.encode(data.x, data.edge_index)
    z_np = ops.convert_to_numpy(z)
    z_tsne = TSNE(n_components=2).fit_transform(z_np)
    y_np = ops.convert_to_numpy(data.y)

    plt.figure(figsize=(8, 8))
    # Robustly determine classes from actual label array
    num_classes = len(np.unique(y_np))
    for i in range(num_classes):
        plt.scatter(
            z_tsne[y_np == i, 0],
            z_tsne[y_np == i, 1],
            s=20,
            color=colors[i % len(colors)],
            label=f"Class {i}",
        )
    plt.axis("off")
    plt.legend(loc="best")
    plt.show()


colors = [
    "#ffc0cb", "#bada55", "#008080", "#420420", "#7fe5f0", "#065535", "#ffd700"
]
plot_points(test_data, colors)